# Jalon 3 — A, B et un agent malveillant

**Situation.** A envoie à B des objets étiquetés par un modèle MR-Sort. Un agent malveillant ajoute, pour chaque objet de A, un objet étiqueté par **un autre modèle MR-Sort**. B reçoit le mélange, sans savoir qui a produit quoi, et doit retrouver le modèle de A.

Ce notebook avance par sous-tâches :
- **T1** : génération du scénario (cette section) ;
- T2 à T5 : algorithme de B, identification du modèle de A, évaluation, exemple ;
- T6 à T8 : stratégie de brouillage.

## T1. Génération du scénario

`make_scenario` produit d'un côté **ce que voit B** (`X`, `y`), de l'autre **ce que B ne voit pas** et qui sert à l'évaluation : la source de chaque objet, les deux modèles et un jeu de test étiqueté par le modèle de A.

| Paramètre | Défaut |
|---|---|
| critères `n` | 6 |
| catégories `p` | 3 |
| objets de A `n_A` | 300 |
| objets du malveillant | la moitié de ceux de A (`ratio_malveillant = 0.5`, voir T3) |
| loi des performances `x` | uniforme sur $[0,1]^n$ pour A et pour le malveillant |
| modèle du malveillant | tiré avec le même générateur que celui de A |

In [ ]:
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
import lincs, lincs.classification as C

print("lincs", lincs.__version__)

### Règle MR-Sort

Fonctions reprises du jalon 2 (`02_jalon2_final`) : `predict` applique la règle MR-Sort en numpy, en float32 comme lincs, avec des poids bruts et un seuil de majorité égal à 1. Un modèle est un dictionnaire `profiles` (forme $(p-1, n)$), `raw_weights`, `weights` (somme 1) et `lmbda`, avec `raw_weights = weights / lmbda`.

In [ ]:
def concordance(model, X):
    # C[k, h] = somme des poids (bruts) des critères où x_k >= b^h, en float32 comme lincs
    X32 = np.asarray(X, dtype=np.float32)
    W32 = np.asarray(model["raw_weights"], dtype=np.float32)
    return np.stack([(X32 >= b.astype(np.float32)).astype(np.float32) @ W32 for b in model["profiles"]],
                    axis=1).astype(float)

def predict(model, X):
    return np.cumprod(concordance(model, X) >= 1., axis=1).sum(axis=1)       # nb de frontières franchies

### Modèle MR-Sort aléatoire

- **Poids** : Dirichlet(1, …, 1), donc positifs de somme 1.
- **λ** : uniforme dans $[0{,}5 ;\, 1[$. En dessous de 0,5, deux coalitions disjointes pourraient franchir la même frontière.
- **Profils** : pour chaque critère, $p-1$ valeurs tirées dans $[0,1]$ puis triées, ce qui impose $b^1 \le b^2 \le \dots$.
- **Contrôle de qualité** : un modèle tiré au hasard peut être dégénéré (presque tous les objets dans une seule catégorie). On le **rejette** si sa plus petite catégorie contient moins de `min_part_categorie` des objets d'un grand tirage de contrôle (10 000 points). Les deux sources passent par le même filtre : elles restent symétriques.

*Pourquoi rejeter :* avec une catégorie quasi vide, prédire une constante suffit pour être précis, les étiquettes des deux sources se contredisent à peine, et il reste trop peu d'exemples pour apprendre les profils de cette catégorie.

In [ ]:
LOIS_X = {"uniforme": lambda rng, k, n: rng.random((k, n))}       # loi des performances : (rng, nb d'objets, n) -> (k, n)

def draw_model(n, p, rng):
    w = rng.dirichlet(np.ones(n))
    lmbda = rng.uniform(0.5, 1.0)
    return {"profiles": np.sort(rng.random((p - 1, n)), axis=0), "weights": w, "raw_weights": w / lmbda, "lmbda": lmbda}

def category_shares(model, p, loi_x, rng, n_control=10_000):
    X = LOIS_X[loi_x](rng, n_control, model["profiles"].shape[1])
    return np.bincount(predict(model, X), minlength=p) / n_control

def random_model(n, p, rng, loi_x="uniforme", min_part_categorie=0.10, max_essais=1000):
    # modèle MR-Sort aléatoire dont chaque catégorie contient au moins min_part_categorie des objets
    for essai in range(1, max_essais + 1):
        model = draw_model(n, p, rng)
        shares = category_shares(model, p, loi_x, rng)
        if shares.min() >= min_part_categorie:
            return dict(model, shares=shares, essais=essai)
    raise RuntimeError(f"aucun modèle valide en {max_essais} essais (min_part_categorie={min_part_categorie}, "
                       f"n={n}, p={p}) : seuil trop élevé ?")

### Configuration et scénario

Sources aléatoires indépendantes (`SeedSequence`) : le modèle de A, le modèle du malveillant, les $x$ de chaque source, le mélange et le jeu de test ne partagent pas leur générateur. Changer une option ne décale donc pas les autres tirages.

Le jeu de test suit la loi de A et est étiqueté par le modèle de A : c'est la référence de la qualité de B. Les lignes sont **mélangées** pour que la position ne trahisse pas la source.

In [ ]:
@dataclass
class ScenarioConfig:
    n: int = 6                              # critères
    p: int = 3                              # catégories
    n_A: int = 300                          # objets envoyés par A
    ratio_malveillant: float = 0.5          # objets du malveillant = ratio * n_A
    loi_x: str = "uniforme"                 # loi des performances de A (et du jeu de test)
    loi_x_malveillant: str | None = None    # None : même loi que A
    min_part_categorie: float = 0.10        # contrôle de qualité des modèles tirés (0 : désactivé)
    n_test: int = 1000                      # taille du jeu de test étiqueté par A

    @property
    def n_M(self):
        return int(round(self.ratio_malveillant * self.n_A))

@dataclass
class Scenario:
    config: ScenarioConfig
    seed: int
    X: np.ndarray                           # ce que voit B : performances (N, n)
    y: np.ndarray                           # ce que voit B : catégories (N,)
    source: np.ndarray                      # caché à B : 0 = A, 1 = malveillant
    model_A: dict
    model_M: dict
    X_test: np.ndarray                      # jeu de test uniforme, étiqueté par le modèle de A
    y_test: np.ndarray

def make_scenario(config=None, seed=0):
    cfg = config or ScenarioConfig()
    rngs = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(7)]
    r_mA, r_mM, r_xA, r_xM, r_mix, r_test, _ = rngs
    loi_M = cfg.loi_x_malveillant or cfg.loi_x

    mA = random_model(cfg.n, cfg.p, r_mA, cfg.loi_x, cfg.min_part_categorie)
    mM = random_model(cfg.n, cfg.p, r_mM, loi_M, cfg.min_part_categorie)

    XA = LOIS_X[cfg.loi_x](r_xA, cfg.n_A, cfg.n)
    XM = LOIS_X[loi_M](r_xM, cfg.n_M, cfg.n)
    X = np.vstack([XA, XM])
    y = np.concatenate([predict(mA, XA), predict(mM, XM)])
    source = np.concatenate([np.zeros(cfg.n_A, int), np.ones(cfg.n_M, int)])

    order = r_mix.permutation(len(X))                                    # B ne doit pas voir l'ordre d'émission
    X_test = LOIS_X[cfg.loi_x](r_test, cfg.n_test, cfg.n)
    return Scenario(cfg, seed, X[order], y[order], source[order], mA, mM, X_test, predict(mA, X_test))

### Exemple

Un scénario avec les paramètres par défaut. Le dernier tableau est exactement ce que reçoit B : des performances et une catégorie, sans la source.

In [ ]:
sc = make_scenario(seed=0)
print(f"{len(sc.X)} objets reçus par B : {(sc.source == 0).sum()} de A, {(sc.source == 1).sum()} du malveillant")

parts = pd.DataFrame({
    "A": np.bincount(sc.y[sc.source == 0], minlength=3) / (sc.source == 0).sum(),
    "malveillant": np.bincount(sc.y[sc.source == 1], minlength=3) / (sc.source == 1).sum(),
    "B voit (mélange)": np.bincount(sc.y, minlength=3) / len(sc.y)}, index=[f"catégorie {h}" for h in range(3)])
display(parts.round(3))
print("essais nécessaires pour tirer les modèles : A", sc.model_A["essais"], "| malveillant", sc.model_M["essais"])

vu_par_B = pd.DataFrame(sc.X, columns=[f"c{i + 1}" for i in range(sc.X.shape[1])]).assign(categorie=sc.y)
display(vu_par_B.head(8).round(3))

### Tests de T1

1. reproductibilité (même graine → mêmes données) ;
2. étiquettes cohérentes avec le modèle de la source ;
3. modèles valides (poids, λ, profils croissants) ;
4. équilibre des catégories, contrôle de qualité effectif ;
5. mélange : tailles des sources et ordre mélangé ;
6. indiscernabilité : la loi des $x$ ne dépend pas de la source ;
7. accord entre `predict` (numpy) et lincs sur ces modèles ;
8. le seuil trop élevé lève bien une erreur.

In [ ]:
def same(a, b):
    return all(np.array_equal(getattr(a, f), getattr(b, f)) for f in ("X", "y", "source", "X_test", "y_test"))

def valid_model(m, n, p):
    return (np.isclose(m["weights"].sum(), 1) and (m["weights"] >= 0).all() and 0.5 <= m["lmbda"] < 1
            and m["profiles"].shape == (p - 1, n) and (np.diff(m["profiles"], axis=0) >= 0).all()
            and ((0 <= m["profiles"]) & (m["profiles"] <= 1)).all())

cfg = ScenarioConfig()
seeds = range(30)
scs = [make_scenario(cfg, s) for s in seeds]

# 1. reproductibilité, et graines différentes -> données différentes
assert same(make_scenario(cfg, 5), make_scenario(cfg, 5))
assert not same(make_scenario(cfg, 5), make_scenario(cfg, 6))

for s in scs:
    A, M = s.source == 0, s.source == 1
    # 2. les étiquettes viennent bien du modèle de la source (et le jeu de test, du modèle de A)
    assert (predict(s.model_A, s.X[A]) == s.y[A]).all() and (predict(s.model_M, s.X[M]) == s.y[M]).all()
    assert (predict(s.model_A, s.X_test) == s.y_test).all()
    # 3. modèles valides
    assert valid_model(s.model_A, cfg.n, cfg.p) and valid_model(s.model_M, cfg.n, cfg.p)
    # 4. aucune catégorie sous le seuil (sur le tirage de contrôle du modèle)
    assert s.model_A["shares"].min() >= cfg.min_part_categorie and s.model_M["shares"].min() >= cfg.min_part_categorie
    # 5. tailles des sources, valeurs dans [0, 1], catégories valides
    assert A.sum() == cfg.n_A and M.sum() == cfg.n_M and s.X.shape == (cfg.n_A + cfg.n_M, cfg.n)
    assert s.X.min() >= 0 and s.X.max() <= 1 and set(np.unique(s.y)) <= set(range(cfg.p))
print("tests 1 à 5 : ok sur", len(scs), "scénarios")

# 5 (suite). le mélange est mélangé : la première moitié ne contient pas que A
frac_A_first_half = np.mean([(s.source[:len(s.source) // 2] == 0).mean() for s in scs])
attendu_A = cfg.n_A / (cfg.n_A + cfg.n_M)                             # part de A dans l'ensemble des lignes
print(f"part de A dans la première moitié des lignes : {frac_A_first_half:.3f} (attendu {attendu_A:.3f})")
assert abs(frac_A_first_half - attendu_A) < 0.05

# 6. indiscernabilité : test de Kolmogorov-Smirnov, par critère, A contre malveillant
pvals = np.array([[stats.ks_2samp(s.X[s.source == 0][:, i], s.X[s.source == 1][:, i]).pvalue
                   for i in range(cfg.n)] for s in scs])
print(f"KS : p-valeur moyenne {pvals.mean():.3f} (attendu ~0.5), part de p < 0.05 : {(pvals < 0.05).mean():.3f} (attendu ~0.05)")
assert (pvals < 0.05).mean() < 0.12

# 4 (suite). catégories des modèles tirés, sur tous les scénarios
shares = np.array([s.model_A["shares"] for s in scs] + [s.model_M["shares"] for s in scs])
print("part de catégorie minimale : min", shares.min().round(3), "| moyenne", shares.min(axis=1).mean().round(3))
print("essais de rejet : moyenne", np.mean([s.model_A["essais"] for s in scs]).round(2),
      "| max", max(s.model_A["essais"] for s in scs))

In [ ]:
# 7. accord numpy / lincs sur un modèle tiré au hasard
s = scs[0]
problem = C.Problem([C.Criterion(f"c{i + 1}", C.Criterion.RealValues(C.Criterion.PreferenceDirection.increasing, 0., 1.))
                     for i in range(cfg.n)], [C.Category(f"cat{h}") for h in range(cfg.p)])
m = s.model_A
lincs_model = C.Model(problem,
    [C.AcceptedValues(C.AcceptedValues.RealThresholds([float(m["profiles"][h, i]) for h in range(cfg.p - 1)]))
     for i in range(cfg.n)],
    [C.SufficientCoalitions(C.SufficientCoalitions.Weights([float(w) for w in m["raw_weights"]]))
     for _ in range(cfg.p - 1)])
alts = C.Alternatives(problem, [C.Alternative(f"a{k}", [C.Performance(C.Performance.Real(float(v))) for v in row])
                                for k, row in enumerate(s.X)])
C.classify_alternatives(problem, lincs_model, alts)
accord = np.mean([a.category_index for a in alts.alternatives] == predict(m, s.X))
print("accord numpy / lincs :", accord)
assert accord == 1.0

# 8. un seuil impossible lève une erreur claire
try:
    random_model(6, 3, np.random.default_rng(0), min_part_categorie=0.4, max_essais=50)
    raise SystemExit("erreur attendue")
except RuntimeError as e:
    print("erreur attendue levée :", e)

## T2. Algorithme de B : inv-2-MR-Sort accéléré

B reçoit le mélange `(X, y)` et ne sait ni quelle part vient de A, ni quelle part vient du malveillant. Il applique l'algorithme du jalon 2 (alternance apprentissage lincs par segment / réaffectation, voir `02_jalon2_final`), avec trois changements pour le rendre rapide :

1. **Arrêt sur un seuil de changements** : on s'arrête quand moins de `TOL_CHANGES` (1 %) des objets changent de segment, ou après `MAX_ITER` itérations. Avec `TOL_CHANGES = 0`, on retrouve l'arrêt strict du jalon 2.
2. **Départs en parallèle** (`joblib`, backend `loky`) : les 4 départs (1 guidé + 3 aléatoires) sont indépendants, chacun tourne dans son processus. `loky` sérialise les fonctions du notebook avec cloudpickle.
3. **Modèles purement numpy en sortie** : les objets lincs (`_lincs`, `_problem`) ne sont pas renvoyés par les processus. Ils ne servent qu'à l'apprentissage.

Le partage des objets ambigus (`share_ambiguous`) est abandonné : le jalon 2 a conclu qu'il ne faisait pas mieux (phase 1).

B reste **aveugle** : l'algorithme ne connaît ni la proportion des sources ni le niveau de bruit.

In [ ]:
import re
import time
from pathlib import Path

import yaml
from joblib import Parallel, delayed

ITER_SECONDS   = 1.0     # temps lincs par apprentissage, pendant les itérations (pour 6 critères / 3 catégories)
POLISH_SECONDS = 4.0     # temps lincs de l'apprentissage final (polish), même référence
MAX_ITER       = 10      # nombre maximal d'itérations par départ (20 au jalon 2 ; 10 retenu par la validation ci-dessous)
TOL_CHANGES    = 0.01    # arrêt quand moins de 1 % des objets changent de segment (0 : arrêt strict du jalon 2)
INITS          = ("errors", "random", "random", "random")   # départs : 1 guidé + 3 aléatoires
MIN_FRAC       = 0.10    # taille minimale d'un segment pour apprendre son modèle

In [ ]:
def n_params(n, p):
    return n * (p - 1) + n + 1

def budget(base_seconds, n, p):
    # temps alloué à lincs, proportionnel au nombre de paramètres du modèle (6 critères / 3 catégories = référence)
    return base_seconds * n_params(n, p) / n_params(6, 3)

def make_problem(n, p):
    crits = [C.Criterion(f"c{i + 1}", C.Criterion.RealValues(C.Criterion.PreferenceDirection.increasing, 0., 1.))
             for i in range(n)]
    return C.Problem(crits, [C.Category(f"cat{h}") for h in range(p)])

def to_alts(problem, X, y=None):
    return C.Alternatives(problem, [
        C.Alternative(f"a{k}", [C.Performance(C.Performance.Real(float(v))) for v in X[k]],
                      None if y is None else int(y[k])) for k in range(len(X))])

def learn_mrsort(X, y, p, seed=0, seconds=2.0, models=9, patience=30):
    # heuristique weights-profiles-breed de lincs : un seul modèle MR-Sort
    problem = make_problem(X.shape[1], p)
    pre = C.PreprocessedLearningSet(problem, to_alts(problem, X, y))
    mbl = C.LearnMrsortByWeightsProfilesBreed.ModelsBeingLearned(pre, models, int(seed))
    init = C.InitializeProfilesForProbabilisticMaximalDiscriminationPowerPerCriterion(pre, mbl)
    weights = C.OptimizeWeightsUsingGlop(pre, mbl)
    profiles = C.ImproveProfilesWithAccuracyHeuristicOnCpu(pre, mbl)
    breed = C.ReinitializeLeastAccurate(mbl, init, weights, count=models // 2)
    term = C.TerminateWhenAny([C.TerminateAtAccuracy(mbl, len(X)), C.TerminateAfterSeconds(seconds),
                               C.TerminateAfterIterationsWithoutProgress(mbl, patience)])
    m = C.LearnMrsortByWeightsProfilesBreed(pre, mbl, init, weights, profiles, breed, term).perform()
    B = np.array([[np.inf if av.real_thresholds.thresholds[h] is None else av.real_thresholds.thresholds[h]
                   for av in m.accepted_values] for h in range(p - 1)])          # (p-1, n)
    W = np.array(m.sufficient_coalitions[0].weights.criterion_weights)           # seuil de majorité = 1 (lincs)
    return {"profiles": B, "raw_weights": W, "weights": W / W.sum(), "lmbda": 1. / W.sum()}

def margins(model, X, y):
    # marge de x dans sa catégorie y : positive si bien classé, d'autant plus grande que le classement est solide
    Cn = (concordance(model, X) - 1.) / model["raw_weights"].sum()          # C(x, b^h) - lambda, poids normalisés
    m = np.full(len(X), np.inf)
    low = y >= 1                                                             # doit franchir b^{h-1}
    m[low] = np.minimum(m[low], Cn[low, y[low] - 1] + 1e-12)
    up = np.where(y <= Cn.shape[1] - 1)[0]                                   # ne doit pas franchir b^h
    m[up] = np.minimum(m[up], -Cn[up, y[up]] - 1e-12)
    m[np.isinf(m)] = 1.
    return m

In [ ]:
def scores(models, X, y):
    E = np.stack([np.abs(predict(m, X) - y) for m in models], axis=1)     # (N, 2) écart de catégorie
    M = np.stack([margins(m, X, y) for m in models], axis=1)              # (N, 2) marge
    return E, M

def reassign(models, X, y, seg):
    E, M = scores(models, X, y)
    key = E - 1e-3 * np.clip(M, -1, 1)               # lexicographique : l'écart domine, la marge départage
    best = key.argmin(axis=1)
    rows = np.arange(len(X))
    move = key[rows, best] < key[rows, seg] - 1e-12  # strictement mieux expliqué ailleurs
    return np.where(move, best, seg), E

def fit_segments(X, y, p, seg, rng, seconds):
    models = []
    for k in (0, 1):
        idx = np.where(seg == k)[0]
        size_min = int(max(10, MIN_FRAC * len(X)))
        if len(idx) < size_min:                       # segment trop petit : tirage aléatoire
            idx = rng.choice(len(X), size=size_min, replace=False)
        models.append(learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=seconds))
    return models

def init_partition(X, y, p, how, rng, seconds):
    if how == "random":
        return rng.integers(0, 2, len(X))
    single = learn_mrsort(X, y, p, seed=int(rng.integers(1e9)), seconds=seconds)
    return (predict(single, X) != y).astype(int)      # "errors" : erreurs du modèle unique -> segment 1

def learn_2_mrsort(X, y, p, init="errors", seed=0, max_iter=MAX_ITER, tol_changes=TOL_CHANGES,
                   iter_seconds=ITER_SECONDS, polish_seconds=POLISH_SECONDS):
    # un départ : alternance apprentissage / réaffectation, puis polish. Renvoie le meilleur état rencontré.
    rng = np.random.default_rng(seed)
    n = X.shape[1]
    t_iter, t_polish = budget(iter_seconds, n, p), budget(polish_seconds, n, p)
    seg = init_partition(X, y, p, init, rng, t_iter)
    best, history = None, []

    for it in range(max_iter):
        models = fit_segments(X, y, p, seg, rng, t_iter)                   # apprentissage
        new_seg, E = reassign(models, X, y, seg)                           # réaffectation
        score = (E.min(axis=1) == 0).mean()                                # expliqués par au moins un modèle
        changes = (new_seg != seg).mean()
        sizes = np.bincount(new_seg, minlength=2)
        history.append(dict(iteration=it + 1, expliques=score, petit_segment=sizes.min() / sizes.sum(),
                            ambigus=(E == 0).all(axis=1).mean(), changements=changes))
        if best is None or score > best["score"]:
            best = dict(score=score, models=models, seg=new_seg.copy(), E=E)
        if changes <= tol_changes:                                         # arrêt : partition (presque) stable
            break
        seg = new_seg

    # polish : apprentissage plus long sur la partition retenue
    seg = best["seg"]
    polished = []
    for k in (0, 1):
        idx = np.where(seg == k)[0]
        old = best["models"][k]
        if len(idx) < 10:
            polished.append(old)
            continue
        new = learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=t_polish)
        better = (predict(new, X[idx]) == y[idx]).mean() >= (predict(old, X[idx]) == y[idx]).mean()
        polished.append(new if better else old)
    new_seg, E = reassign(polished, X, y, seg)
    score = (E.min(axis=1) == 0).mean()
    if score >= best["score"]:
        best.update(score=score, models=polished, seg=new_seg, E=E)

    best["history"] = pd.DataFrame(history)
    best["init"] = init
    return best

def learn_B(X, y, p, inits=INITS, seed=0, n_jobs=None, **kwargs):
    # lance un départ par élément de `inits`, en parallèle (n_jobs=1 : séquentiel), et garde le meilleur
    n_jobs = len(inits) if n_jobs is None else n_jobs
    jobs = (delayed(learn_2_mrsort)(X, y, p, init=how, seed=seed * 100 + r, **kwargs) for r, how in enumerate(inits))
    runs = Parallel(n_jobs=n_jobs, backend="loky")(jobs)
    return max(runs, key=lambda r: r["score"]), runs

### Choix techniques

| Choix | Pourquoi |
|---|---|
| `learn_B` renvoie `(best, runs)` | `best` contient `models`, `seg`, `E`, `score`, `init`, `history`. `runs` garde tous les départs pour le diagnostic. |
| Graines `seed * 100 + r` (comme le jalon 2) | Les départs de `learn_B` et de la version séquentielle du jalon 2 partent des mêmes graines. |
| `learn_mrsort` ne renvoie plus `_lincs` ni `_problem` | Ces objets lincs ne se sérialisent pas entre processus, et T3 et T4 n'utilisent que le numpy. L'accord numpy / lincs de `predict` est testé en T1. |
| `n_jobs=1` désactive le parallélisme | Sert de référence séquentielle pour la validation et le débogage. |
| `fit_segments` sans `shared` | `share_ambiguous` n'est plus géré (voir plus haut). |

### Essai sur un scénario du jalon 3

Un scénario T1 (300 objets de A, 300 du malveillant) passé à `learn_B`. La vérité terrain n'est utilisée que pour **afficher** la pureté de la partition : B ne la voit pas.

In [ ]:
sc = make_scenario(seed=0)
t = time.time()
best, runs = learn_B(sc.X, sc.y, sc.config.p, seed=0)
temps = time.time() - t

sizes = np.bincount(best["seg"], minlength=2)
accord = max((best["seg"] == sc.source).mean(), (best["seg"] != sc.source).mean())     # segments définis à une permutation près
print(f"{temps:.0f} s | départ retenu : {best['init']} | objets expliqués par au moins un modèle : {best['score']:.3f}")
print(f"tailles des segments : {sizes.tolist()} | accord segments / sources : {accord:.3f}")
for r in runs:
    print(f"   départ {r['init']:6s} -> {r['score']:.3f}  ({len(r['history'])} itérations)")

### Validation sur les fichiers du jalon 2

**Question.** Les accélérations (arrêt à 1 %, moins d'itérations, départs en parallèle) dégradent-elles le résultat par rapport à l'algorithme du jalon 2 ?

**Protocole** (celui du jalon 2) : 10 % des objets en test, jamais utilisés pour décider. Mesures : `deux_test` (part des objets de test bien classés par au moins un des deux modèles), `erreur_proportion` (écart entre la taille du petit segment trouvé et la taille attendue, lue dans le nom du fichier), `temps`. Fichiers : `n0600_rep50-50_noise00` de 4 dossiers (le cas 50/50 sans bruit, proche du jalon 3). Mêmes graines pour toutes les variantes.

**Variantes**
- *jalon 2* : séquentiel, `MAX_ITER = 20`, arrêt strict (`TOL_CHANGES = 0`) ;
- *parallèle, 4 départs* avec `TOL_CHANGES = 0,01` et `MAX_ITER` = 8, 10, 20 ;
- *parallèle, 2 départs* (1 guidé + 1 aléatoire).

**Critère de choix** : la configuration la plus rapide dont `deux_test` ne baisse pas de plus de 1 à 2 points par rapport au jalon 2. L'objectif de temps est de moins de 60 s pour 600 objets.

Par défaut la cellule relit `t2_validation.csv`. Passer `RUN_T2 = True` pour relancer les mesures (long : plusieurs dizaines de minutes).

In [ ]:
DATA_DIR = Path("../data/Datasets-Step2")

def load_dataset(csv_path):
    csv_path = Path(csv_path)
    pb = yaml.safe_load((csv_path.parent / "problem.yml").read_text())
    criteria = [str(c["name"]) for c in pb["criteria"]]
    categories = [str(c["name"]) for c in pb["ordered_categories"]]
    df = pd.read_csv(csv_path)
    y = df["category"].astype(str).map({c: h for h, c in enumerate(categories)})
    return df[criteria].to_numpy(float), y.to_numpy(int), len(categories)

def split(N, seed, test_frac=0.10):
    rng = np.random.default_rng(10_000 + seed)
    test = np.zeros(N, bool)
    test[rng.choice(N, round(test_frac * N), replace=False)] = True
    return test

def two_models_accuracy(models, X, y):
    return np.stack([predict(m, X) == y for m in models]).any(axis=0).mean()

VARIANTES = {
    "jalon 2 (séquentiel, 20 it., arrêt strict)": dict(n_jobs=1, max_iter=20, tol_changes=0.0),
    "parallèle, 4 départs, 8 it.":  dict(max_iter=8),
    "parallèle, 4 départs, 10 it.": dict(max_iter=10),
    "parallèle, 4 départs, 20 it.": dict(max_iter=20),
    "parallèle, 2 départs, 10 it.": dict(max_iter=10, inits=("errors", "random")),
}

def bench_t2(files, seeds, variantes, out_csv=None):
    rows = []
    for f in files:
        X, y, p = load_dataset(f)
        rep = re.search(r"rep(\d+)-(\d+)", f.name)
        attendu = min(int(rep[1]), int(rep[2])) / 100
        for s in seeds:
            test = split(len(X), s)
            for nom, kw in variantes.items():
                t = time.time()
                best, _ = learn_B(X[~test], y[~test], p, seed=s, **kw)
                sizes = np.bincount(best["seg"], minlength=2)
                rows.append(dict(dossier=f"{f.parent.parent.name}/{f.parent.name}", fichier=f.stem, graine=s, variante=nom,
                                 deux_train=best["score"], deux_test=two_models_accuracy(best["models"], X[test], y[test]),
                                 erreur_proportion=abs(sizes.min() / sizes.sum() - attendu), depart=best["init"],
                                 iterations=len(best["history"]), temps=time.time() - t))
                if out_csv:
                    pd.DataFrame(rows).to_csv(out_csv, index=False)            # sauvegarde au fil de l'eau
                r = rows[-1]
                print(f"{r['dossier']} graine={s} {nom:45s} test={r['deux_test']:.3f} temps={r['temps']:.0f} s", flush=True)
    return pd.DataFrame(rows)

T2_FILES = [DATA_DIR / d / "n0600_rep50-50_noise00.csv"
            for d in ("type_a/cr06_ca03", "type_a/cr08_ca04", "type_b/cr06_ca03", "type_b/cr08_ca04")]
RUN_T2 = False
t2 = (bench_t2(T2_FILES, seeds=[0], variantes=VARIANTES, out_csv="../results/t2_validation.csv")
      if RUN_T2 else pd.read_csv("../results/t2_validation.csv"))

In [ ]:
ref = list(VARIANTES)[0]
ordre = [v for v in VARIANTES if v in set(t2.variante)]
res = t2.groupby("variante")[["deux_test", "erreur_proportion", "iterations", "temps"]].agg(["mean", "std"]).reindex(ordre)
res.columns = [f"{a} ({b})" for a, b in res.columns]
ecart = (t2.pivot_table(index=["fichier", "dossier", "graine"], columns="variante", values="deux_test")
         .sub(t2.pivot_table(index=["fichier", "dossier", "graine"], columns="variante", values="deux_test")[ref], axis=0))
res["écart deux_test vs jalon 2"] = ecart.mean().reindex(ordre)
display(res.round(3))
display(t2.pivot_table(index=["dossier"], columns="variante", values="deux_test")[ordre].round(3))

### Conclusion de la validation

**On retient 4 départs en parallèle, `MAX_ITER = 10`, `TOL_CHANGES = 0,01`.**

- **Temps** : environ 41 s en moyenne par fichier de 600 objets (29 à 53 s), contre 418 s au jalon 2 (122 à 1081 s). L'objectif de 60 s est tenu, à environ 10 fois plus rapide.
- **Précision** : `deux_test` vaut 0,954 contre 0,967 au jalon 2, soit 1,2 point de moins (écart apparié : écart-type de 3,2 points). C'est la seule variante rapide à rester dans les 1 à 2 points visés.
- **Le nombre de départs ne change pas le temps** en parallèle (2 départs : 39 s, 4 départs : 41 s), mais 2 départs perdent 5,4 points. On garde 4 départs.
- **`MAX_ITER`** : 8 itérations perdent 5 points, et 20 itérations perdent 3,7 points tout en coûtant plus cher (67 s). L'ordre n'est pas monotone : avec 4 fichiers, une graine et 60 objets de test (1,7 point par objet), les écarts de 3 à 5 points sont du même ordre que le bruit. Ces mesures montrent que 10 itérations ne dégradent pas nettement le jalon 2, pas que 10 est strictement meilleur que 8 ou 20.

**Limites à garder en tête**
- `erreur_proportion` se dégrade : 0,071 pour la configuration retenue contre 0,033 au jalon 2. Le petit segment est moins bien dimensionné. Cela pèse sur T3 (identification du modèle de A).
- `deux_train` baisse aussi (0,974 contre 0,996) : moins d'itérations, un ajustement moins poussé.
- lincs est lui-même multithread (OpenMP) : un apprentissage seul utilise environ 5 cœurs. Quatre départs parallèles se partagent donc les cœurs, et chaque apprentissage, limité à 1 s, fait moins de travail (précision d'apprentissage de 0,75 seul contre 0,71 en parallèle). Le gain de temps vient donc surtout des itérations en moins et de l'arrêt à 1 %.
- Un essai de la variante à 8 itérations a pris 486 s sur un fichier (contre 46 à 50 s ailleurs), sans cause identifiée : à surveiller.

## T3. Identifier le modèle de A

Après T2, B a **deux modèles** et un rangement de chaque objet dans l'un des deux segments, mais les segments « 0 » et « 1 » n'ont pas de sens : B ne sait pas lequel est celui de A.

**Un problème d'identifiabilité.** Avec les choix de T1 (même générateur de modèles, même loi des $x$) et un ratio de 1, les deux sources sont **parfaitement symétriques** : choisir A serait un pile ou face, même avec une séparation parfaite. L'information n'est pas dans les données. Il faut une asymétrie. On exploite celle de l'énoncé : le malveillant « peut » envoyer une donnée additionnelle pour chaque donnée de A, donc il envoie **au plus autant** que A. S'il envoie moins, **le plus grand segment est celui de A**.

| Choix | Décision |
|---|---|
| Asymétrie exploitée | ratio du malveillant < 1 : le plus grand segment est supposé celui de A |
| Ratio par défaut | 0,5 (A représente 67 % de ce que reçoit B). Balayage en T4 sur {1,0 ; 0,85 ; 0,7 ; 0,5} |
| Indicateur de confiance | écart relatif de taille $(n_{grand} - n_{petit}) / N$, **taille seulement** |
| Décision | confiance $\ge$ seuil : B choisit le modèle du plus grand segment ; sinon **abstention** : B renvoie les deux modèles |
| Seuil | **provisoire** (`SEUIL_CONFIANCE`), calibré en T4 sur des scénarios de calibration distincts de ceux du rapport final |
| Évaluation | deux métriques : **oracle** (qualité de la séparation) et **identifié** (qualité du choix) |

*Pourquoi la taille seulement :* rien ne rend le modèle de A mieux ajusté que celui du malveillant. Ajouter la qualité d'ajustement créerait un signal artificiel.

*Pourquoi l'abstention :* quand les tailles sont proches, B annoncerait une certitude qu'il n'a pas. Le cas ratio = 1 est le pire cas pour B et le meilleur pour le malveillant : il doit s'abstenir.

In [ ]:
from dataclasses import dataclass

SEUIL_CONFIANCE = 0.10      # PROVISOIRE : sera calibré en T4. Écart de taille des segments, en part de N.

@dataclass
class Decision:
    abstention: bool                  # True : B ne choisit pas et renvoie les deux modèles
    confiance: float                  # (n_grand - n_petit) / N, entre 0 et 1
    indice_segment_A: int | None      # segment supposé être celui de A (None si abstention)
    modele_A: dict | None             # modèle supposé être celui de A (None si abstention)
    modeles: list | None              # les deux modèles (renseigné seulement en cas d'abstention)
    tailles: tuple                    # (taille du segment 0, taille du segment 1)
    seuil: float

def identify_A(result_B, seuil=SEUIL_CONFIANCE):
    # result_B : meilleur résultat de learn_B (clés `seg` et `models`).
    # Le plus grand segment est supposé être celui de A (ratio du malveillant < 1) ; en cas d'égalité, le segment 0.
    seg, models = result_B["seg"], result_B["models"]
    tailles = np.bincount(seg, minlength=2)
    confiance = abs(int(tailles[0]) - int(tailles[1])) / len(seg)
    if confiance < seuil:
        return Decision(True, confiance, None, None, list(models), (int(tailles[0]), int(tailles[1])), seuil)
    k = int(np.argmax(tailles))
    return Decision(False, confiance, k, models[k], None, (int(tailles[0]), int(tailles[1])), seuil)

### Évaluation : séparation et identification

Deux métriques, mesurées sur le jeu de test du scénario (objets uniformes, **étiquetés par le modèle de A**) :

| Mesure | Définition | Ce qu'elle dit |
|---|---|---|
| `oracle` | précision du meilleur des deux modèles appris | qualité de la **séparation** (T2) : B a-t-il appris le modèle de A, quel que soit le choix ? |
| `identifie` | précision du modèle choisi par B ; **NaN en cas d'abstention** | qualité du **choix** (T3) : B a-t-il rendu le bon des deux ? |
| `bon_segment` | le segment choisi contient majoritairement des objets de A (d'après `source`) ; NaN en cas d'abstention | l'identification est correcte, au sens de la partition |
| `grand_segment_est_A` | même test pour le plus grand segment, **abstention ou non** | l'hypothèse « le plus grand segment est A » est-elle juste ? Sert à calibrer le seuil |
| `accord` | part des objets bien rangés, à une permutation près | pureté de la partition |

`identifie` vaut NaN en cas d'abstention : l'abstention est comptée à part (`abstention`, taux en T4), et la précision moyenne des cas où B se prononce se calcule avec `nanmean`. On ne remplace pas NaN par l'`oracle`, ce qui masquerait le coût de l'abstention.

`identifie` $\le$ `oracle` toujours. Si `oracle` est bon et `identifie` mauvais, l'échec vient du choix (T3) ; si `oracle` est mauvais, il vient de la séparation (T2).

In [ ]:
def model_accuracy(model, X, y):
    return float((predict(model, X) == y).mean())

def evaluate_identification(scenario, result_B, decision):
    # Compare le résultat de B à la vérité terrain du scénario (que B ne voit pas).
    seg, models, source = result_B["seg"], result_B["models"], scenario.source
    acc = [model_accuracy(m, scenario.X_test, scenario.y_test) for m in models]
    part_A = [float((source[seg == k] == 0).mean()) if (seg == k).any() else np.nan for k in (0, 1)]   # part d'objets de A par segment
    grand = int(np.argmax(np.bincount(seg, minlength=2)))
    choisi = decision.indice_segment_A
    return dict(
        abstention=bool(decision.abstention), confiance=float(decision.confiance),
        oracle=max(acc), oracle_indice=int(np.argmax(acc)),
        identifie=np.nan if decision.abstention else acc[choisi],
        bon_segment=np.nan if decision.abstention else float(part_A[choisi] > 0.5),
        grand_segment_est_A=float(part_A[grand] > 0.5),
        accord=float(max((seg == source).mean(), (seg != source).mean())),
        taille_grand=int(max(decision.tailles)), taille_petit=int(min(decision.tailles)),
        n_A=int((source == 0).sum()), n_M=int((source == 1).sum()))

### Tests de T3 (sans lincs)

Résultats de B **synthétiques** : on construit un `result_B` à partir de la vérité terrain d'un scénario T1 (partition parfaite ou volontairement fausse), ce qui rend les tests déterministes et instantanés.

1. ratio 0,5, partition parfaite : B choisit le segment de A, `oracle` = `identifie` = 1 ;
2. mêmes données avec les numéros de segments **permutés** : même identification (les numéros n'ont pas de sens) ;
3. ratio 1 : tailles égales, abstention, `identifie` = NaN, `oracle` = 1 ;
4. seuil 0 : B choisit toujours, même à égalité ;
5. frontière du seuil : une confiance exactement égale au seuil donne un choix ;
6. limite : si le malveillant envoie **plus** que A (ratio 2), l'hypothèse est fausse et B se trompe de modèle, avec `oracle` = 1 ;
7. partition aléatoire : `accord` proche de 0,5 et `oracle` inférieur à 1.

In [ ]:
def perfect_result(sc, swap=False):
    # résultat de B si la séparation était parfaite ; swap : numéros de segments permutés
    seg = (1 - sc.source) if swap else sc.source.copy()
    models = [sc.model_M, sc.model_A] if swap else [sc.model_A, sc.model_M]
    return dict(seg=seg, models=models)

sc5 = make_scenario(ScenarioConfig(ratio_malveillant=0.5), seed=11)         # 300 contre 150 : confiance = 1/3

# 1. séparation parfaite : le choix est le bon
r = perfect_result(sc5); d = identify_A(r); e = evaluate_identification(sc5, r, d)
assert not d.abstention and d.indice_segment_A == 0 and d.modele_A is sc5.model_A and d.modeles is None
assert np.isclose(d.confiance, 1 / 3) and d.tailles == (300, 150)
assert e["oracle"] == e["identifie"] == 1.0 and e["bon_segment"] == 1.0 and e["accord"] == 1.0

# 2. numéros de segments permutés : même identification
r = perfect_result(sc5, swap=True); d = identify_A(r); e = evaluate_identification(sc5, r, d)
assert d.indice_segment_A == 1 and d.modele_A is sc5.model_A
assert e["identifie"] == 1.0 and e["bon_segment"] == 1.0

# 3. ratio 1 : tailles égales, abstention
sc1 = make_scenario(ScenarioConfig(ratio_malveillant=1.0), seed=11)
r = perfect_result(sc1); d = identify_A(r); e = evaluate_identification(sc1, r, d)
assert d.abstention and d.confiance == 0 and d.modele_A is None and d.indice_segment_A is None and len(d.modeles) == 2
assert np.isnan(e["identifie"]) and np.isnan(e["bon_segment"]) and e["oracle"] == 1.0

# 4. seuil 0 : B choisit toujours, même à égalité (le segment 0)
d = identify_A(r, seuil=0.0)
assert not d.abstention and d.indice_segment_A == 0

# 5. frontière : confiance == seuil -> choix ; juste au-dessus -> abstention
r = perfect_result(sc5)
assert not identify_A(r, seuil=1 / 3).abstention and identify_A(r, seuil=1 / 3 + 1e-9).abstention

# 6. limite : si le malveillant envoie plus que A, le plus grand segment est celui du malveillant
sc2 = make_scenario(ScenarioConfig(ratio_malveillant=2.0), seed=11)
r = perfect_result(sc2); d = identify_A(r); e = evaluate_identification(sc2, r, d)
assert not d.abstention and d.modele_A is sc2.model_M
assert e["bon_segment"] == 0.0 and e["grand_segment_est_A"] == 0.0 and e["oracle"] == 1.0 and e["identifie"] < e["oracle"]

# 7. partition aléatoire
rng = np.random.default_rng(0)
r = dict(seg=rng.integers(0, 2, len(sc5.X)), models=[sc5.model_A, sc5.model_M])
e = evaluate_identification(sc5, r, identify_A(r))
assert 0.45 < e["accord"] < 0.62
print("tests T3 : ok (7 groupes)")

### Essai réel sur des scénarios T1

Chaîne complète : `make_scenario` → `learn_B` → `identify_A` → `evaluate_identification`, sur quelques scénarios (ce n'est pas le benchmark de T4). Les graines 1 à 3 en ratio 0,5, et la graine 1 en ratio 1,0 (même modèles de A et du malveillant qu'en ratio 0,5) pour montrer l'abstention attendue.

Chaque scénario coûte environ 30 à 40 s. Par défaut la cellule relit `t3_essais.csv`. Passer `RUN_T3 = True` pour relancer.

In [ ]:
ESSAIS_T3 = [(1, 0.5), (2, 0.5), (3, 0.5), (1, 1.0)]       # (graine, ratio du malveillant)

def essai_t3(seed, ratio, seuil=SEUIL_CONFIANCE):
    sc = make_scenario(ScenarioConfig(ratio_malveillant=ratio), seed)
    t = time.time()
    best, _ = learn_B(sc.X, sc.y, sc.config.p, seed=seed)
    temps = time.time() - t
    ev = evaluate_identification(sc, best, identify_A(best, seuil))
    return dict(graine=seed, ratio=ratio, **ev, temps=temps)

RUN_T3 = False
if RUN_T3:
    rows = []
    for s, ratio in ESSAIS_T3:
        rows.append(essai_t3(s, ratio))
        pd.DataFrame(rows).to_csv("../results/t3_essais.csv", index=False)            # sauvegarde au fil de l'eau
        print(rows[-1], flush=True)
t3 = pd.read_csv("../results/t3_essais.csv")
cols = ["graine", "ratio", "n_A", "n_M", "taille_grand", "taille_petit", "confiance", "abstention",
        "oracle", "identifie", "bon_segment", "grand_segment_est_A", "accord", "temps"]
display(t3[cols].round(3))

### Résultats de l'essai réel et limites

| Constat | Chiffres |
|---|---|
| Ratio 0,5 : l'identification est correcte sur les 3 scénarios | `bon_segment` = 1 ; `identifie` = `oracle` = 0,88 / 0,86 / 0,88 |
| Les segments trouvés sont **plus équilibrés que les vrais** | trouvés 285/165, 261/189, 305/145 contre 300/150 réels ; confiance 0,27 / 0,16 / 0,36 contre 0,33 réelle |
| Le scénario d'exemple (graine 0) donnerait une abstention | segments 230/220, confiance 0,02 (accord 0,73) |
| **Ratio 1,0 : B ne s'abstient pas et se trompe** | segments trouvés 365/235 pour 300/300 réels, confiance 0,22 > seuil de 0,10 ; `bon_segment` = 0, `identifie` = 0,44 pour un `oracle` de 0,88 |
| La séparation reste moyenne | `oracle` ≈ 0,88 : même le meilleur des deux modèles classe 12 % du test autrement que le modèle de A |

**À retenir pour T4.** À ratio 1, les tailles trouvées s'écartent de 50/50 de plusieurs dizaines d'objets sans aucun signal : une confiance de 0,22 ne prouve rien, et le seuil provisoire de 0,10 est trop bas. L'indicateur de confiance doit être calibré sur des scénarios à ratio 1, où l'on connaît la bonne réponse (aucune), pour fixer un seuil qui limite les faux choix. Quatre essais ne suffisent pas à le faire : c'est le rôle de T4. Détails dans `docs/jalon3.md`.